# Group 2 — Project 2 (OptiTrack + Delsys + force plates)

Your second capture runs **three instruments on two clocks**. This notebook shows how to (1) load each modality, (2) **synchronize** them, and (3) read off aligned signals.

**Sensors** — Delsys `Ch1` = R lateral gastroc, `Ch2` = L lateral gastroc (each EMG + accel), `Ch13` = **analog SYNC** (Motive record gate). Force plates come **through Motive**, so they and the OptiTrack markers already share the **Motive clock**; only the Delsys is on its own clock.

In [ ]:
!pip install -q delsys pysampled huggingface_hub

### 1 · Get your data

In [ ]:
from huggingface_hub import hf_hub_download
REPO = "praneethnamburimit/immersionlab-pe-mis-groups"
def get(p): return hf_hub_download(REPO, p, repo_type="dataset")
delsys_csv = get("group_2_2/delsys/Trial_1.csv")
fp1_csv    = get("group_2_2/bertec/group2_project2_001_forceplate_1.csv")
fp2_csv    = get("group_2_2/bertec/group2_project2_001_forceplate_2.csv")
markers_csv= get("group_2_2/ot/group2_project2_001.csv")
print("downloaded")

### 2 · Load each modality (raw, on its own clock)
**Delsys** — EMG from the two calf sensors:

In [ ]:
import delsys, numpy as np, matplotlib.pyplot as plt
lf = delsys.Log(delsys_csv)
print("sensors:", lf.sensor_numbers)
emg_R = lf.find(sensor_number=1, as_="sensor")[0].emg   # R lateral gastroc
emg_L = lf.find(sensor_number=2, as_="sensor")[0].emg   # L lateral gastroc
plt.figure(figsize=(12,2.5)); plt.plot(emg_R.t, emg_R()); plt.xlim(30,35); plt.title("R lat-gastroc EMG (Delsys clock)"); plt.show()

**Force plates** — `MotiveLog` (not on pip, so pasted here). `MocapTime` is the Motive clock.

In [ ]:
import csv as _csv, numpy as np, pandas as pd, pysampled
class MotiveLog:
    """Load a Motive force-plate 'Device export' CSV. MocapTime is the OptiTrack clock."""
    def __init__(self, fname):
        hdr = {}
        with open(fname, newline='') as f:
            for rc, row in enumerate(_csv.reader(f)):
                if not row: continue
                if row[0] == 'MocapFrame': break
                v = row[1] if len(row) > 1 else None
                try: hdr[row[0]] = float(v)
                except (TypeError, ValueError): hdr[row[0]] = v
        self.data = pd.read_csv(fname, skiprows=rc).rename(columns=lambda x: x.strip())
        self.sr = hdr['Mocap Rate'] * hdr.get('Mocap Rate Multiple', 1)
    def __getattr__(self, k):
        if k in ('Fx','Fy','Fz','Mx','My','Mz','Cx','Cy','Cz','Tz'):
            return pysampled.Data(self.data[k].values, sr=self.sr)
        raise AttributeError(k)

In [ ]:
fp1 = MotiveLog(fp1_csv); fp2 = MotiveLog(fp2_csv)
print("force-plate sr:", fp1.sr, "Hz")
fz1 = np.abs(np.asarray(fp1.Fz()).ravel()); fp_t = np.arange(len(fz1))/fp1.sr   # Motive clock (starts at 0)
plt.figure(figsize=(12,2.5)); plt.plot(fp_t, fz1); plt.xlim(30,35); plt.title("force plate 1 |Fz| (Motive clock)"); plt.show()

**OptiTrack markers** — the Motive export (markers: L/R foot, hip, shoulder). Column 1 is `Time` (Motive clock).

In [ ]:
import pandas as pd, csv as _c
def load_markers(path):
    with open(path) as f:                               # find the 'Frame, Time (Seconds), X, Y, Z...' header row
        h = next(i for i, row in enumerate(_c.reader(f)) if any('Time (Seconds)' in str(x) for x in row))
    m = pd.read_csv(path, skiprows=h)
    return m.loc[:, ~m.columns.astype(str).str.startswith('Unnamed')]
mk = load_markers(markers_csv)
mk_t = mk['Time (Seconds)'].to_numpy(float)             # Motive clock (starts at 0)
xyz = mk.drop(columns=[c for c in ('Frame', 'Time (Seconds)') if c in mk]).to_numpy(float)
# markers in order: Lfoot, Lhip, Lshoulder, Rfoot, Rhip, Rshoulder (X,Y,Z each)
rfoot_z = xyz[:, 3*3 + 2]                               # Rfoot = 4th marker, vertical (Z)
plt.figure(figsize=(12,2.5)); plt.plot(mk_t, rfoot_z); plt.xlim(30,35); plt.title('Rfoot height (Motive clock)'); plt.show()

### 3 · Synchronize — the key step
Two clocks: the **force plates + markers** are on the **Motive clock** (both start at 0). The **Delsys** runs on its own clock and was started at a different moment. To line them up, the Delsys recorded Motive's **record gate** on an analog channel — find its rising edge and you have the offset between the clocks.

In [ ]:
import numpy as np
def motive_gate_offset(lf):
    """Delsys analog sensor 13, channel 0 = Motive's record gate (HIGH while Motive records).
    Its rising edge is the Delsys-clock time at which Motive (force plates + markers) began at t=0."""
    a = lf.find(sensor_number=13, as_="sensor")[0].analog
    g = np.asarray(a())[:, 0]; t = np.asarray(a.t)
    return float(t[np.argmax(g > 0.5*(g.min()+g.max()))])

In [ ]:
offset = motive_gate_offset(lf)
print(f"Motive started at Delsys t = {offset:.2f} s")
# move ANY Delsys signal onto the Motive clock:   t_motive = t_delsys - offset

### 4 · Load synchronized signals
Shift the Delsys EMG by −offset and it lines up with the force plate on one time axis:

In [ ]:
env = emg_R.bandpass(20,450).envelope()
emg_t = np.asarray(env.t) - offset                 # Delsys EMG, now on the Motive clock
fig, ax = plt.subplots(2,1, figsize=(13,5), sharex=True)
ax[0].plot(fp_t, fz1); ax[0].set_ylabel("|Fz| (N)"); ax[0].set_title("force plate (Motive clock)")
ax[1].plot(emg_t, np.asarray(env()).ravel(), color="tab:orange"); ax[1].set_ylabel("R gastroc EMG env")
ax[1].set_xlabel("Motive time (s)"); ax[1].set_title("Delsys EMG shifted onto the Motive clock — now aligned")
[a.set_xlim(40,70) for a in ax]; plt.tight_layout(); plt.show()

### 5 · Your turn
- Everything is now on the **Motive clock** (force plates, markers as-is; Delsys after `− offset`). Pick your task windows and compare across modalities — e.g. calf EMG vs. ground-reaction force vs. foot height.
- The left calf is `sensor_number=2`; `.acc` on each sensor gives shank acceleration.
- Both plates are loaded (`fz1`, and `fp2`) — sum them for total force, or keep them per-foot.